In [1]:
# This code is run on kaggle.
# The dataset used is splits-updated-223 and msba-group7-ssl-models on kaggle.
# The model stat_dictionary file is on msba-group7-ssl-models.

In [2]:
!pip install d2l==1.0.3 transformers evaluate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.9/58.9 kB 4.0 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of datasets to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of datasets to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.7/111.7 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 110.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 93.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 105.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.6/62.6 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.4/34.4 MB 54.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.0/542.0 kB 35.2 MB/s e

In [3]:
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset, Dataset
from d2l import torch as d2l
import os
import pandas as pd
import torchvision
import torchaudio
from transformers import AdamW, Wav2Vec2Config,Wav2Vec2Processor, Wav2Vec2ForSequenceClassification, AutoConfig, Wav2Vec2ForCTC, Wav2Vec2ForPreTraining,Trainer, TrainingArguments, get_cosine_schedule_with_warmup
from tqdm import tqdm
from transformers.data.data_collator import default_data_collator
from datasets import load_dataset
import torch.optim as optim
import evaluate
import numpy as np

# 加载模型

In [4]:
from safetensors.torch import load_file

config = AutoConfig.from_pretrained("/kaggle/input/msba-group7-ssl-models/old_version/old_version/model/run_2/checkpoint-2176/config.json",num_labels=3)
model = Wav2Vec2ForPreTraining.from_pretrained("facebook/wav2vec2-base", config=config)

state_dict = load_file("/kaggle/input/msba-group7-ssl-models/old_version/old_version/model/run_2/checkpoint-2176/model.safetensors")
model.load_state_dict(state_dict)

processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base-960h",num_labels=3)

pytorch_model.bin:   0%|          | 0.00/380M [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/159 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/163 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.60k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/291 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

In [5]:
from torch.nn import functional as F

class AudioClassifier(nn.Module):
    def __init__(self, num_classes):
        super(AudioClassifier, self).__init__()
        self.wav2vec2 = model 
        self.fc1 = nn.Linear(256, 64)  
        self.fc2 = nn.Linear(64, num_classes) 

    def forward(self, input_values):
        with torch.no_grad():
            outputs = self.wav2vec2(input_values)
            features = outputs.projected_states
        
        x = features.mean(dim=1)  
        
        x = self.fc1(x)
        x = F.relu(x)  
        
        x = self.fc2(x)
        return x

model = AudioClassifier(num_classes=3) 

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

AudioClassifier(
  (wav2vec2): Wav2Vec2ForPreTraining(
    (wav2vec2): Wav2Vec2Model(
      (feature_extractor): Wav2Vec2FeatureEncoder(
        (conv_layers): ModuleList(
          (0): Wav2Vec2GroupNormConvLayer(
            (conv): Conv1d(1, 512, kernel_size=(10,), stride=(5,), bias=False)
            (activation): GELUActivation()
            (layer_norm): GroupNorm(512, 512, eps=1e-05, affine=True)
          )
          (1-4): 4 x Wav2Vec2NoLayerNormConvLayer(
            (conv): Conv1d(512, 512, kernel_size=(3,), stride=(2,), bias=False)
            (activation): GELUActivation()
          )
          (5-6): 2 x Wav2Vec2NoLayerNormConvLayer(
            (conv): Conv1d(512, 512, kernel_size=(2,), stride=(2,), bias=False)
            (activation): GELUActivation()
          )
        )
      )
      (feature_projection): Wav2Vec2FeatureProjection(
        (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (projection): Linear(in_features=512, out_features=7

# 加载数据集

In [6]:
import os
import shutil
from pathlib import Path

def prepare_datasets(
    input_dir='/kaggle/input/splits-updated-223/splits/MELD',
    working_dir="/kaggle/working",
    train_dir="train",
    test_dir="test",
    val_dir="val"
):
    """
    只复制训练集和测试集的音频文件到工作目录，不合并验证集
    """
    try:
        # 创建工作目录下的训练和测试目录
        working_train_dir = os.path.join(working_dir, "train")
        working_test_dir = os.path.join(working_dir, "test")
        working_val_dir = os.path.join(working_dir, "val")

        # 如果目标目录已存在，先删除
        if os.path.exists(working_train_dir):
            shutil.rmtree(working_train_dir)
        if os.path.exists(working_test_dir):
            shutil.rmtree(working_test_dir)
        if os.path.exists(working_val_dir):
            shutil.rmtree(working_val_dir)
            
        os.makedirs(working_train_dir, exist_ok=True)
        os.makedirs(working_test_dir, exist_ok=True)
        os.makedirs(working_val_dir, exist_ok=True)

        # 复制训练集音频文件
        train_files = list(Path(os.path.join(input_dir, train_dir, "audio")).rglob("*.wav"))
        print(f"找到训练集音频文件: {len(train_files)} 个")
        
        # 复制测试集音频文件
        test_files = list(Path(os.path.join(input_dir, test_dir, "audio")).rglob("*.wav"))
        print(f"找到测试集音频文件: {len(test_files)} 个")

        # 复制val集音频文件
        val_files = list(Path(os.path.join(input_dir, val_dir, "audio")).rglob("*.wav"))
        print(f"找到val集音频文件: {len(val_files)} 个")

        # 复制训练集到工作目录
        total_train_files = 0
        for file_path in train_files:
            # 直接获取文件名
            file_name = file_path.name
            target_path = os.path.join(working_train_dir, file_name)
            
            # 复制文件
            shutil.copy2(str(file_path), target_path)
            total_train_files += 1

        # 复制测试集到测试目录
        total_test_files = 0
        for file_path in test_files:
            file_name = file_path.name
            target_path = os.path.join(working_test_dir, file_name)
            shutil.copy2(str(file_path), target_path)
            total_test_files += 1

        # 复制val集到测试目录
        total_val_files = 0
        for file_path in val_files:
            file_name = file_path.name
            target_path = os.path.join(working_val_dir, file_name)
            shutil.copy2(str(file_path), target_path)
            total_val_files += 1

        print(f"成功复制训练集到 {working_train_dir}，共 {total_train_files} 个文件")
        print(f"成功复制测试集到 {working_test_dir}，共 {total_test_files} 个文件")
        print(f"成功复制val集到 {working_val_dir}，共 {total_val_files} 个文件")
        return True

    except Exception as e:
        print(f"处理数据集时出错: {str(e)}")
        return False

if __name__ == "__main__":
    # 使用示例
    prepare_datasets()

找到训练集音频文件: 3856 个
找到测试集音频文件: 1016 个
找到val集音频文件: 435 个
成功复制训练集到 /kaggle/working/train，共 3856 个文件
成功复制测试集到 /kaggle/working/test，共 1016 个文件
成功复制val集到 /kaggle/working/val，共 435 个文件


# 数据预处理

In [7]:
# 三分类
# 数据转换成字典 
class MELDDataset(Dataset):
    def __init__(self, data_dir, processor):
        self.data_dir = data_dir
        self.processor = processor
        self.audio_files = [f for f in os.listdir(data_dir) if f.endswith(".wav")]

    def __len__(self):
        return len(self.audio_files)

    def __getitem__(self, idx):
        file_path = os.path.join(self.data_dir, self.audio_files[idx])
        speech_array, sampling_rate = torchaudio.load(file_path)
        # Resample if necessary
        if sampling_rate != 16000:
            resampler = torchaudio.transforms.Resample(sampling_rate, 16000)
            speech_array = resampler(speech_array)
        inputs = self.processor(
            speech_array.squeeze(), 
            sampling_rate=16000, 
            return_tensors="pt", 
            padding="max_length", 
            max_length=16000 * 3,  #音频长度为3秒
            truncation=True
        )
        
        # label
        label = self.audio_files[idx].split("_")[-1].split(".")[0]
        label_id = self.label_to_id(label)

        # input(.numpy???)
        input_values = inputs["input_values"].squeeze()
        # 归一化
        mean = input_values.mean()
        std = input_values.std()
        normalized_input_values = (input_values - mean) / std
        
        return {
            "input_values": normalized_input_values,  
            "labels": torch.tensor(label_id, dtype=torch.long)
        }
        
    def label_to_id(self, label):
        label_map = {
            "neutral": 2,  
            "joy": 0,    
            "sadness": 1, 
            "anger": 1,   
            "surprise": 0,
            "fear": 1,    
            "disgust": 1  
        }
        return label_map.get(label, -1)

    

  

In [8]:
# dataset and dataloader
train_dataset = MELDDataset(data_dir="/kaggle/working/train", processor=processor)
valid_dataset = MELDDataset(data_dir="/kaggle/working/val", processor=processor)
test_dataset = MELDDataset(data_dir="/kaggle/working/test", processor=processor)
#train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
#valid_loader = DataLoader(valid_dataset, batch_size=8, shuffle=False)
#test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)

# 模型训练

In [9]:
def train(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0
    for batch in tqdm(loader, desc="Training"):
        input_values = batch["input_values"]
        labels = batch["labels"]
        
        optimizer.zero_grad()
        
        outputs = model(input_values)  
        loss = criterion(outputs, labels)  
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    accuracy = correct / total
    return total_loss / len(loader), accuracy

def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for batch in tqdm(loader, desc="Evaluating"):
            input_values = batch["input_values"]
            labels = batch["labels"]
            
            outputs = model(input_values)  
            loss = criterion(outputs, labels)  
            
            total_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    accuracy = correct / total
    return total_loss / len(loader), accuracy


In [10]:
def collate_fn(batch):
    input_values = [item["input_values"] for item in batch]
    labels = [item["labels"] for item in batch]
    input_values = torch.nn.utils.rnn.pad_sequence(input_values, batch_first=True).to(device)  # 移动到 device
    labels = torch.tensor(labels, dtype=torch.long).to(device)  # 移动到 device
    return {
        "input_values": input_values,
        "labels": labels
    }

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, collate_fn=collate_fn)
valid_loader = DataLoader(valid_dataset, batch_size=8, shuffle=False, collate_fn=collate_fn)


from torch.optim.lr_scheduler import ReduceLROnPlateau

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=0.00005, weight_decay=0.01)

scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=2)

In [11]:
# 解冻
for param in model.fc1.parameters():
    param.requires_grad = True

# 其他层
for param in model.wav2vec2.parameters():
    param.requires_grad = True


In [12]:
best_val_accuracy = 0.0
num_epochs = 5
for epoch in range(num_epochs):
    train_loss, train_acc = train(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = evaluate(model, valid_loader, criterion, device)

    scheduler.step(val_acc)
    
    print(f"Epoch {epoch + 1}/{num_epochs}")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
    
    # 保存最佳模型
    if val_acc > best_val_accuracy:
        best_val_accuracy = val_acc
        torch.save(model.state_dict(), "best_audio_classifier.pth")


Evaluating: 100%|██████████| 55/55 [00:08<00:00,  6.67it/s]


Epoch 1/5
Train Loss: 1.0772 | Train Acc: 0.4404
Val Loss: 1.1020 | Val Acc: 0.3931


Evaluating: 100%|██████████| 55/55 [00:07<00:00,  6.91it/s]


Epoch 2/5
Train Loss: 1.0699 | Train Acc: 0.4450
Val Loss: 1.1026 | Val Acc: 0.3931


Evaluating: 100%|██████████| 55/55 [00:08<00:00,  6.79it/s]


Epoch 3/5
Train Loss: 1.0659 | Train Acc: 0.4461
Val Loss: 1.0932 | Val Acc: 0.3977


Evaluating: 100%|██████████| 55/55 [00:08<00:00,  6.82it/s]


Epoch 4/5
Train Loss: 1.0645 | Train Acc: 0.4453
Val Loss: 1.0949 | Val Acc: 0.3977


Evaluating: 100%|██████████| 55/55 [00:08<00:00,  6.76it/s]


Epoch 5/5
Train Loss: 1.0629 | Train Acc: 0.4497
Val Loss: 1.0948 | Val Acc: 0.4000


In [13]:
# 保存最终模型
torch.save(model.state_dict(), "audio_classifier_state_dict_3.pth")
# 保存模型配置文件
config.save_pretrained("/kaggle/working")